<a href="https://colab.research.google.com/github/august-ehrlich/delex-lm/blob/main/CompLing2ProjectFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens;
!pip install torch
!pip install datasets
!pip install --upgrade ipython

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.3/945.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=6dda862257b13f96ef4519e3917ee46b80183e893d4ff416fd8316b796644596
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: decorator
    Found existi

In [ ]:
%load_ext autoreload
%autoreload 2
import datasets
import torch
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens import HookedTransformer
from collections import Counter
from typing import List
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/CL2 Project/Drive Upload/Delex LM')


In [ ]:
class UDSurfaceTokenizer:
    def __init__(self):
        # 1. Define mandatory special tokens for causal modeling
        self.PAD_TOKEN = "<pad>"  # Used to make sequences the same length in a batch
        self.UNK_TOKEN = "<unk>"  # Used if the model sees a bundle it never saw in training
        self.EOS_TOKEN = "<eos>"  # End of sequence (already in your dataset)

        self.special_tokens = [self.PAD_TOKEN, self.UNK_TOKEN, self.EOS_TOKEN]

        # 2. Initialize the vocabulary mappings
        self.vocab_to_id = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.id_to_vocab = {idx: token for idx, token in enumerate(self.special_tokens)}

        self.vocab_size = len(self.vocab_to_id)

    def train_on_corpus(self, corpus: List[str], min_freq: int = 1):
        """
        Reads the training data, finds all unique tokens, and assigns them an ID.
        `corpus` should be a list of your dataset's strings.
        """
        token_counts = Counter()

        # Count frequencies of every space-separated block
        for sentence in corpus:
            tokens = sentence.strip().split()
            token_counts.update(tokens)

        # Add to vocabulary if it meets the minimum frequency threshold
        # (Filtering out rare tokens can help prevent the model from memorizing noise)
        for token, count in token_counts.items():
            if token not in self.vocab_to_id and count >= min_freq:
                idx = len(self.vocab_to_id)
                self.vocab_to_id[token] = idx
                self.id_to_vocab[idx] = token

        self.vocab_size = len(self.vocab_to_id)
        print(f"Vocabulary built! Total unique tokens: {self.vocab_size}")

    def encode(self, text: str, add_eos: bool = False) -> List[int]:
        """
        Converts a string of UD bundles/literals into a list of integer IDs.
        """
        tokens = text.strip().split()

        # Map token to ID, fallback to <unk> if it doesn't exist
        token_ids = [self.vocab_to_id.get(token, self.vocab_to_id[self.UNK_TOKEN]) for token in tokens]

        if add_eos:
            token_ids.append(self.vocab_to_id[self.EOS_TOKEN])

        return token_ids

    def decode(self, token_ids: List[int]) -> str:
        """
        Converts a list of integer IDs back into the readable UD string format.
        """
        tokens = [self.id_to_vocab.get(idx, self.UNK_TOKEN) for idx in token_ids]
        return " ".join(tokens)

    def save(self, filepath: str):
        """Saves the vocabulary dictionary to disk."""
        import json
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.vocab_to_id, f, ensure_ascii=False, indent=2)

    def load(self, filepath: str):
        """Loads a pre-computed vocabulary dictionary."""
        import json
        with open(filepath, 'r', encoding='utf-8') as f:
            self.vocab_to_id = json.load(f)
        self.id_to_vocab = {int(idx): token for token, idx in self.vocab_to_id.items()}
        self.vocab_size = len(self.vocab_to_id)